# 06 - Perfilamiento de Clústeres y Business Insights

## Objetivo
Cubrir los entregables 5 y 6 del proyecto:
- **Entregable 5:** Resumen estadístico que describa cada segmento (ej. "El grupo X gasta un 50% más que el promedio")
- **Entregable 6:** Gráfico 3D con los segmentos coloreados y recomendación de negocio para cada uno

Fuente: `bigdata.default.gold_customer_segments` (generada por el notebook 05)_

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings("ignore")

# Leer tabla Gold con clusters ya asignados (generada por notebook 05)
df_segments = spark.table("bigdata.default.gold_customer_segments")
print("Clientes en gold_customer_segments:", df_segments.count())
df_segments.show(5)

In [0]:
# Convertir a Pandas para visualización
pdf = df_segments.toPandas()
print(f"Clientes cargados: {len(pdf):,}")
print(f"Clústeres encontrados: {sorted(pdf['cluster'].unique())}")

## Entregable 5 — Perfilamiento Estadístico de Clústeres

Calculamos las métricas RFM promedio de cada clúster y las comparamos
contra el promedio global para describir cada segmento con precisión.


In [0]:
# Promedios globales de referencia
global_recency  = pdf["recency"].mean()
global_freq     = pdf["frequency"].mean()
global_monetary = pdf["monetary"].mean()

print("Promedios globales de referencia:")
print(f"  Recency   : {global_recency:.1f} días")
print(f"  Frequency : {global_freq:.1f} compras")
print(f"  Monetary  : ${global_monetary:.2f}")

In [0]:
# Perfil RFM por clúster
resumen = pdf.groupby("cluster").agg(
    clientes       = ("id_cliente",  "count"),
    recency_media  = ("recency",     "mean"),
    freq_media     = ("frequency",   "mean"),
    monetary_media = ("monetary",    "mean"),
    monetary_total = ("monetary",    "sum")
).round(2).reset_index()

resumen["pct_clientes"]      = (resumen["clientes"] / resumen["clientes"].sum() * 100).round(1)
resumen["monetary_vs_avg_%"] = ((resumen["monetary_media"] / global_monetary - 1) * 100).round(1)
resumen["recency_vs_avg_%"]  = ((resumen["recency_media"]  / global_recency  - 1) * 100).round(1)
resumen["freq_vs_avg_%"]     = ((resumen["freq_media"]     / global_freq     - 1) * 100).round(1)

print("=" * 75)
print("  PERFIL RFM POR CLÚSTER")
print("=" * 75)
print(resumen[[
    "cluster", "clientes", "pct_clientes",
    "recency_media", "recency_vs_avg_%",
    "freq_media",    "freq_vs_avg_%",
    "monetary_media","monetary_vs_avg_%"
]].to_string(index=False))

In [0]:
# Resumen ejecutivo en lenguaje de negocio (estilo enunciado)
print("=" * 65)
print("  RESUMEN EJECUTIVO POR CLÚSTER")
print("=" * 65)

for _, row in resumen.iterrows():
    c   = int(row["cluster"])
    n   = int(row["clientes"])
    pct = row["pct_clientes"]
    r   = row["recency_media"]
    f   = row["freq_media"]
    m   = row["monetary_media"]
    dm  = row["monetary_vs_avg_%"]
    df_ = row["freq_vs_avg_%"]
    dr  = row["recency_vs_avg_%"]

    print(f"\n  Clúster {c}  ({n:,} clientes — {pct}% del total)")
    print(f"  {'─'*55}")
    print(f"  Recency media  : {r:.1f} días  ({dr:+.1f}% vs promedio global)")
    print(f"  Frequency media: {f:.1f} compras ({df_:+.1f}% vs promedio global)")
    print(f"  Monetary media : ${m:,.2f}  ({dm:+.1f}% vs promedio global)")

In [0]:
# Asignar etiquetas de negocio basadas en el perfil real de cada clúster
def asignar_etiqueta(row):
    alta_freq     = row["freq_media"]     >= global_freq
    alto_monetary = row["monetary_media"] >= global_monetary
    alta_recency  = row["recency_media"]  >= global_recency  # más días = más inactivo

    if alta_freq and alto_monetary and not alta_recency:
        return "VIP / Champions"
    elif alta_recency and (alta_freq or alto_monetary):
        return "En Riesgo"
    elif not alta_recency and not alta_freq and not alto_monetary:
        return "Nuevos / Potenciales"
    else:
        return "Regulares / Ocasionales"

resumen["segmento"] = resumen.apply(asignar_etiqueta, axis=1)

# Mapear etiqueta al DataFrame principal
etiqueta_map    = dict(zip(resumen["cluster"], resumen["segmento"]))
pdf["segmento"] = pdf["cluster"].map(etiqueta_map)

print("Etiquetas asignadas por clúster:")
print(resumen[["cluster", "segmento", "clientes", "pct_clientes",
               "monetary_media", "monetary_vs_avg_%"]].to_string(index=False))

## Entregable 6 — Visualización y Storytelling

Gráficos 2D y 3D con los segmentos coloreados,
seguidos de recomendaciones de negocio para cada uno.

In [0]:
# Paleta de colores por segmento
segmentos   = resumen["segmento"].tolist()
colores_map = {seg: cm.tab10(i / len(segmentos)) for i, seg in enumerate(segmentos)}

# --- Gráfica 2D: Frequency vs Monetary ---
plt.figure(figsize=(11, 7))
for seg in segmentos:
    mask = pdf["segmento"] == seg
    plt.scatter(pdf.loc[mask, "frequency"],
                pdf.loc[mask, "monetary"],
                label=seg, alpha=0.6, s=40,
                color=colores_map[seg])

plt.xlabel("Frequency (número de compras)", fontsize=12)
plt.ylabel("Monetary (gasto total $)", fontsize=12)
plt.title("Segmentación de Clientes — Frecuencia vs Gasto", fontsize=14, fontweight="bold")
plt.legend(title="Segmento", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# --- Gráfica 3D: Espacio RFM completo ---
fig = plt.figure(figsize=(13, 8))
ax  = fig.add_subplot(111, projection="3d")

for seg in segmentos:
    mask = pdf["segmento"] == seg
    ax.scatter(pdf.loc[mask, "recency"],
               pdf.loc[mask, "frequency"],
               pdf.loc[mask, "monetary"],
               label=seg, alpha=0.6, s=30,
               color=colores_map[seg])

ax.set_xlabel("Recency (días)", fontsize=11, labelpad=10)
ax.set_ylabel("Frequency (compras)", fontsize=11, labelpad=10)
ax.set_zlabel("Monetary ($)", fontsize=11, labelpad=10)
ax.set_title("Segmentación RFM 3D — K-Means", fontsize=14, fontweight="bold")
ax.legend(title="Segmento", fontsize=9)
plt.tight_layout()
plt.show()

In [0]:
# --- Distribución de clientes por segmento ---
dist         = pdf["segmento"].value_counts().reset_index()
dist.columns = ["segmento", "clientes"]
colores_bar  = [colores_map[s] for s in dist["segmento"]]

plt.figure(figsize=(9, 5))
bars = plt.bar(dist["segmento"], dist["clientes"],
               color=colores_bar, edgecolor="white")

for bar, val in zip(bars, dist["clientes"]):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 5,
             f"{val:,}", ha="center", va="bottom",
             fontsize=11, fontweight="bold")

plt.title("Distribución de Clientes por Segmento", fontsize=14, fontweight="bold")
plt.xlabel("Segmento", fontsize=12)
plt.ylabel("Número de Clientes", fontsize=12)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# --- Recomendaciones de negocio por segmento ---
recomendaciones = {
    "VIP / Champions": [
        "Programa de fidelización exclusivo: acceso anticipado a nuevos productos y beneficios premium.",
        "Invitarlos como embajadores de marca o panel de feedback de producto.",
        "Comunicación personalizada — evitar saturarlos con promociones genéricas."
    ],
    "En Riesgo": [
        "Campaña de reactivación con cupón de descuento por tiempo limitado.",
        "Email personalizado recordando su historial ('Te echamos de menos').",
        "Ofrecer novedades relacionadas con sus compras anteriores."
    ],
    "Nuevos / Potenciales": [
        "Secuencia de onboarding: correos de bienvenida con guías y recomendaciones.",
        "Descuento en segunda compra para incentivar la recompra.",
        "Mostrar reseñas y casos de éxito para generar confianza."
    ],
    "Regulares / Ocasionales": [
        "Promociones estacionales en fechas clave para estimular compras.",
        "Programa de puntos acumulables para aumentar frecuencia.",
        "Recomendaciones personalizadas basadas en historial de compra."
    ]
}

print("=" * 65)
print("  RECOMENDACIONES DE NEGOCIO POR SEGMENTO")
print("=" * 65)

for seg, acciones in recomendaciones.items():
    if seg not in pdf["segmento"].unique():
        continue

    n       = pdf[pdf["segmento"] == seg].shape[0]
    pct     = n / len(pdf) * 100
    rfm_seg = pdf[pdf["segmento"] == seg][["recency", "frequency", "monetary"]].mean()
    dm      = (rfm_seg["monetary"] / global_monetary - 1) * 100

    print(f"\n{'─'*65}")
    print(f"  {seg.upper()}  ({n:,} clientes — {pct:.1f}% del total)")
    print(f"{'─'*65}")
    print(f"  Recency  : {rfm_seg['recency']:.1f} días sin comprar")
    print(f"  Frequency: {rfm_seg['frequency']:.1f} compras únicas")
    print(f"  Monetary : ${rfm_seg['monetary']:,.2f}  ({dm:+.1f}% vs promedio global)")
    print(f"\n  Acciones recomendadas:")
    for i, accion in enumerate(acciones, 1):
        print(f"    {i}. {accion}")

print(f"\n{'='*65}")
print(f"  Total clientes analizados : {len(pdf):,}")
print(f"  Monetary total            : ${pdf['monetary'].sum():,.2f}")
print(f"{'='*65}")